# Task 6: сравнение word2vec моделей

Решение сделано по аналогии с `lab6.ipynb` для пунктов со скрина.

In [3]:
!pip install -q gensim scipy pandas

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from gensim.models import KeyedVectors

## Загрузка моделей и датасета

In [4]:
from pathlib import Path
import urllib.request
import zipfile
import gensim.downloader as api

# Google News model (без gdown/wget)
w_gn = api.load("word2vec-google-news-300")

# British National Corpus model
br_zip_url = "http://vectors.nlpl.eu/repository/20/0.zip"
br_zip_path = Path("0.zip")
br_model_path = Path("model.bin")

if not br_model_path.exists():
    if not br_zip_path.exists():
        urllib.request.urlretrieve(br_zip_url, br_zip_path)

    with zipfile.ZipFile(br_zip_path, "r") as zf:
        zf.extract("model.bin")

w_br = KeyedVectors.load_word2vec_format(str(br_model_path), binary=True)

# wordsim датасет: читаем напрямую по URL
df = pd.read_csv(
    "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Text_Processing/Task_6/wordsim_similarity_goldstandard.txt",
    sep="\t",
    header=None,
)
df.columns = ["first", "second", "score"]
df.head(3)

[==================================================] 100.0% 1662.8/1662.8MB downloaded


,first,second,score
0,tiger,cat,7.35
1,tiger,tiger,10.00
2,plane,car,5.77


In [5]:
def cosine_distance(v1, v2):
    return 1 - np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def get_gn_token(word):
    if word in w_gn:
        return word
    cap = word.capitalize()
    if cap in w_gn:
        return cap
    return None

## 1) Косинусное расстояние между `television` и `media`

In [6]:
gn_q1 = w_gn.distance("television", "media")
br_q1 = w_br.distance("television_NOUN", "media_NOUN")

print("GN:", round(gn_q1, 3))
print("BR:", round(br_q1, 3))

GN: 0.541
BR: 0.487


## 2) Лишнее слово для набора `television media bread cucumber`

In [7]:
gn_odd = w_gn.doesnt_match(["television", "media", "bread", "cucumber"])
br_odd = w_br.doesnt_match(["television_NOUN", "media_NOUN", "bread_NOUN", "cucumber_NOUN"])

print("GN:", gn_odd)
print("BR:", br_odd.replace("_NOUN", "").lower())

GN: cucumber
BR: media


## 3) Косинусное расстояние между векторами предложений

Предложения:
- `cat has nine lives`
- `chain is only as strong as its weakest link`

Для GN используем только слова, которые есть в словаре модели.

In [8]:
sent1 = "cat has nine lives".split()
sent2 = "chain is only as strong as its weakest link".split()

sent1_known = [t for t in sent1 if t in w_gn]
sent2_known = [t for t in sent2 if t in w_gn]

vec1 = np.mean([w_gn[t] for t in sent1_known], axis=0)
vec2 = np.mean([w_gn[t] for t in sent2_known], axis=0)

gn_sent_dist = cosine_distance(vec1, vec2)
print("GN:", round(gn_sent_dist, 3))

GN: 0.742


## 4) Корреляция Спирмена на подвыборке `word_sim[13:113]`

Берем только пары, для которых есть векторы в BR как существительные (`_NOUN`).
Считаем Spearman между оценками моделей и `score` аннотаторов.

In [9]:
df_sub = df.iloc[13:113].copy()

gn_scores = []
br_scores = []
human_scores = []
removed = 0

for _, row in df_sub.iterrows():
    w1 = row["first"]
    w2 = row["second"]

    br_w1 = f"{w1.lower()}_NOUN"
    br_w2 = f"{w2.lower()}_NOUN"

    # Удаляем пару, если в BR нет хотя бы одного существительного.
    if br_w1 not in w_br or br_w2 not in w_br:
        removed += 1
        continue

    gn_w1 = get_gn_token(w1)
    gn_w2 = get_gn_token(w2)
    if gn_w1 is None or gn_w2 is None:
        continue

    gn_scores.append(w_gn.similarity(gn_w1, gn_w2))
    br_scores.append(w_br.similarity(br_w1, br_w2))
    human_scores.append(row["score"])

gn_corr = spearmanr(gn_scores, human_scores).statistic
br_corr = spearmanr(br_scores, human_scores).statistic

print("GN Spearman:", round(gn_corr, 3))
print("BR Spearman:", round(br_corr, 3))
print("Removed pairs:", removed)

GN Spearman: 0.69
BR Spearman: 0.649
Removed pairs: 3
